# AI Frequency

AI 调用频率 & 时机的特征分析。

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# 输入文件路径
data_dir = Path('../../data/pickle')
timeline_file = data_dir / 'timeline.pkl'
ai_events_file = data_dir / 'ai_events.pkl'
user_msgs_file = data_dir / 'user_msgs.pkl'

# 输出文件路径
output_dir = Path('../../data/analysis/ai_freq')
ai_freq_metrics_file = output_dir / 'ai_freq_metrics.csv'
ai_call_density_file = output_dir / 'ai_call_density.csv'
output_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# 加载预处理数据
timeline = pd.read_pickle(timeline_file)
ai_events = pd.read_pickle(ai_events_file)
user_msgs = pd.read_pickle(user_msgs_file)

# 按被试和物品分组计算指标
grouped = ai_events.groupby(['participant_id', 'item_name', 'trial_index'])

## 调用总次数

In [4]:
# 1. 调用总次数
call_count = grouped.size().reset_index(name='ai_call_count')

## 调用密度

In [5]:
# 2. 调用密度（每30秒窗口）
def compute_density(df, total_duration=300, window=30):
    times = df['click_time'].values
    bins = np.arange(0, total_duration + window, window)
    counts, _ = np.histogram(times, bins=bins)
    return counts

density_list = []
for (pid, item, trial), group in grouped:
    counts = compute_density(group)
    for i, cnt in enumerate(counts):
        density_list.append({
            'participant_id': pid,
            'item_name': item,
            'trial_index': trial,
            'window_start': i * 30,
            'window_end': (i+1) * 30,
            'call_count_in_window': cnt
        })
density_df = pd.DataFrame(density_list)

## 调用间隔稳定性

In [6]:
# 3. 调用间隔稳定性（标准差）
def interval_std(times):
    if len(times) < 2:
        return np.nan
    intervals = np.diff(times)
    return np.std(intervals)

interval_std_series = grouped['click_time'].agg(interval_std).reset_index(name='ai_interval_std')

## 首次 & 末次调用时间

In [7]:
# 4. 首次调用时间
first_call_time = grouped['click_time'].min().reset_index(name='first_ai_time')

# 末次调用时间
last_call_time = grouped['click_time'].max().reset_index(name='last_ai_time')

# 首次到末次之间调用的密度（次数 / 时间）
def density_between_first_last(df, first_time, last_time):
    if len(df) < 2:
        return 0
    times_between = df[(df['click_time'] > first_time) & (df['click_time'] < last_time)].shape[0]
    return times_between / (last_time - first_time) if last_time > first_time else 0

density_first_last = []
for (pid, item, trial), group in grouped:
    first_time = group['click_time'].min()
    last_time = group['click_time'].max()
    density = density_between_first_last(group, first_time, last_time)
    density_first_last.append({
        'participant_id': pid,
        'item_name': item,
        'trial_index': trial,
        'density_first_last': density
    })
density_first_last_df = pd.DataFrame(density_first_last)

## 首次调用前生成想法数量

In [8]:
# 5. 首次调用前生成想法数量
def pre_first_call_ideas(pid, item, trial, first_time):
    user_in_this_trial = user_msgs[
        (user_msgs['participant_id'] == pid) &
        (user_msgs['item_name'] == item) &
        (user_msgs['trial_index'] == trial) &
        (user_msgs['time'] < first_time)
    ]
    return len(user_in_this_trial)

pre_ideas = []
for row in first_call_time.itertuples():
    cnt = pre_first_call_ideas(row.participant_id, row.item_name, row.trial_index, row.first_ai_time)
    pre_ideas.append({
        'participant_id': row.participant_id,
        'item_name': row.item_name,
        'trial_index': row.trial_index,
        'pre_first_call_ideas': cnt
    })
pre_first_df = pd.DataFrame(pre_ideas)

## 调用阶段分布

In [9]:
# 6. 调用阶段分布（可自定义阶段：前期、中期、后期）
def stage_distribution(times, total_duration=300):
    if len(times) == 0:
        return {}
    # 定义三个等长阶段
    stage1 = (0, 100)
    stage2 = (100, 200)
    stage3 = (200, 300)
    count1 = np.sum((times >= stage1[0]) & (times < stage1[1]))
    count2 = np.sum((times >= stage2[0]) & (times < stage2[1]))
    count3 = np.sum((times >= stage3[0]) & (times < stage3[1]))
    return {'stage1_count': count1, 'stage2_count': count2, 'stage3_count': count3}

stage_list = []
for (pid, item, trial), group in grouped:
    stages = stage_distribution(group['click_time'].values)
    stage_list.append({
        'participant_id': pid,
        'item_name': item,
        'trial_index': trial,
        **stages
    })
stage_df = pd.DataFrame(stage_list)

In [10]:
# 合并所有指标
ai_freq_metrics = call_count.merge(interval_std_series, on=['participant_id', 'item_name', 'trial_index'], how='left') \
                             .merge(first_call_time, on=['participant_id', 'item_name', 'trial_index'], how='left') \
                             .merge(last_call_time, on=['participant_id', 'item_name', 'trial_index'], how='left') \
                             .merge(density_first_last_df, on=['participant_id', 'item_name', 'trial_index'], how='left') \
                             .merge(pre_first_df, on=['participant_id', 'item_name', 'trial_index'], how='left') \
                             .merge(stage_df, on=['participant_id', 'item_name', 'trial_index'], how='left')

# ===== 补全所有正式轮次（trial_index=1~4）的被试-轮次组合 =====
# 从 user_msgs 中提取所有正式轮次的唯一组合（确保覆盖所有有想法的被试和轮次）
formal_trials = [1, 2, 3, 4]
all_combos = user_msgs[user_msgs['trial_index'].isin(formal_trials)][
    ['participant_id', 'item_name', 'trial_index']
].drop_duplicates()

# 左连接，补全所有组合
ai_freq_complete = all_combos.merge(
    ai_freq_metrics,
    on=['participant_id', 'item_name', 'trial_index'],
    how='left'
)

# 填充数值列（无调用时设为0）
fill_cols = {
    'ai_call_count': 0,
    'stage1_count': 0,
    'stage2_count': 0,
    'stage3_count': 0,
}
ai_freq_complete.fillna(value=fill_cols, inplace=True)

# 确保 ai_call_count 为整数类型
ai_freq_complete['ai_call_count'] = ai_freq_complete['ai_call_count'].astype(int)

# 其他列（first_ai_time, ai_interval_std, pre_first_call_ideas）保留 NaN，因为它们无定义

# 重新排序
ai_freq_metrics = ai_freq_complete.sort_values(['participant_id', 'trial_index'])
density_df = density_df.sort_values(['participant_id', 'trial_index', 'window_start'])

# 保存结果
ai_freq_metrics.to_csv(ai_freq_metrics_file, index=False)
density_df.to_csv(ai_call_density_file, index=False)